In [1]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import pandas as pd

In [2]:
save_path = "../AutogluonModels/TimeSeriesPredictor"
df = pd.read_csv("../data/processed/datas_cleaned.csv")
df["item_id"] = df["region"] + "_" + df["size_rank"].astype(str)
df["timestamp"] = pd.to_datetime((df["year"] - 543).astype(str))

In [3]:
ts_df = TimeSeriesDataFrame.from_data_frame(
    df[["item_id", "timestamp", "value"]],
    id_column="item_id",
    timestamp_column="timestamp",
)

In [4]:
predictor = TimeSeriesPredictor(
    prediction_length=2, target="value", eval_metric="MASE", path=save_path
).fit(ts_df, presets="medium_quality")

Beginning AutoGluon training...
AutoGluon will save models to 'd:\Mini Project\Term Project\AutogluonModels\TimeSeriesPredictor'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          20
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       5.28 GB / 15.73 GB (33.6%)
Disk Space Avail:   636.99 GB / 953.85 GB (66.8%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selecti

In [5]:
predictor.evaluate(ts_df)

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


{'MASE': -1.7073858155087338}

In [6]:
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")  # จัดรูปแบบตัวเลข
predictions = predictor.predict(ts_df, model="Chronos2").reset_index()
print(predictions["mean"])

0     2,703,553,024
1     2,506,160,128
2    17,322,387,456
3    17,072,662,528
4    37,160,153,088
5    36,365,185,024
6     7,463,370,240
7     6,605,911,040
8     7,452,573,696
9     7,530,031,104
10   18,985,029,632
11   19,440,279,552
12    3,501,608,704
13    3,211,142,656
14    3,770,087,424
15    3,406,664,192
16    3,738,893,568
17    3,678,545,408
18    2,758,758,912
19    2,463,179,776
20    1,818,068,480
21    1,759,191,808
22    2,335,028,480
23    2,255,055,104
24   15,061,064,704
25   13,353,355,264
26   16,288,894,976
27   16,596,432,896
28   30,600,658,944
29   28,167,045,120
Name: mean, dtype: float32


In [7]:
predictions["year_th"] = predictions["timestamp"].dt.year + 543
predictions[["region", "size_rank"]] = predictions["item_id"].str.rsplit(
    "_", n=1, expand=True
)
predictions["size_rank"] = predictions["size_rank"].astype(int)

In [8]:
actual = df[["item_id", "year", "value", "region", "size_rank"]].copy()
actual["year_th"] = actual["year"]
actual["type"] = "actual"

In [9]:
pred_export = predictions[
    ["item_id", "year_th", "region", "size_rank", "mean", "0.1", "0.9"]
].copy()
pred_export.columns = [
    "item_id",
    "year_th",
    "region",
    "size_rank",
    "value",
    "lower",
    "upper",
]
pred_export["type"] = "forecast"

actual_export = actual[["item_id", "year_th", "region", "size_rank", "value"]].copy()
actual_export["lower"] = None
actual_export["upper"] = None
actual_export["type"] = "actual"

combined = pd.concat([actual_export, pred_export], ignore_index=True)
combined.to_csv("../data/predict/predictions.csv", index=False)
print("✅ บันทึกเป็น predictions.csv แล้ว")

✅ บันทึกเป็น predictions.csv แล้ว


C:\Users\UNS_CT\AppData\Local\Temp\ipykernel_34380\4235570968.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([actual_export, pred_export], ignore_index=True)
